In [2]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx

In [3]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a fitted CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)
    
    for node in bn.nodes:
        cpd = bn.cpds[node]
        num_node_states = cpd.shape[0]
        num_parent_configs = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)
        parents = list(bn.structure.predecessors(node))
        
        if len(parents) == 0:
            probs = data[node].map(cpd.iloc[:, 0]).values
        else:
            cpd_series = cpd.unstack()
            parent_order = sorted(parents)
            lookup_cols = parent_order + [node]
            lookup_df = data[lookup_cols]
            lookup_keys = [tuple(x) for x in lookup_df.to_numpy()]
            probs = cpd_series.loc[lookup_keys].values
            
        log_likelihood += np.sum(np.log(probs + 1e-9))
    
    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [4]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [5]:
def discretize_data(df):
    discretised_df = pd.DataFrame(index=df.index)
    TARGET_BINS = 3

    for col in df.columns:
        if df[col].isna().all():
            discretised_df[col] = "Missing"
            continue
        
        no_unique = df[col].nunique()
    
        if no_unique <= 1:
            discretised_df[col] = "Constant"
            continue
    
        if no_unique <= TARGET_BINS:
            discretised_df[col] = df[col].rank(method='dense') - 1
        else:
            try:
                discretised_df[col] = pd.qcut(df[col], q=TARGET_BINS, labels=False, duplicates='drop')
            except (ValueError, TypeError):
                try:
                    discretised_df[col] = pd.cut(df[col], bins=TARGET_BINS, labels=False, include_lowest=True)
                except (ValueError, TypeError):
                    discretised_df[col] = pd.qcut(df[col].rank(method='first'), q=TARGET_BINS, labels=False, duplicates='drop')

        if pd.api.types.is_numeric_dtype(discretised_df[col]):
            valid_values = sorted(discretised_df[col].dropna().unique())
            num_states = len(valid_values)
            
            if num_states == 2:
                label_map = {valid_values[0]: "Low", valid_values[1]: "High"}
            elif num_states == 3:
                label_map = {valid_values[0]: "Low", valid_values[1]: "Medium", valid_values[2]: "High"}
            elif num_states == 4:
                label_map = {valid_values[0]: "VeryLow", valid_values[1]: "Low", valid_values[2]: "High", valid_values[3]: "VeryHigh"}
            else:
                label_map = {val: f"State{int(val)}" for val in valid_values}
            
            discretised_df[col] = discretised_df[col].map(label_map)
            
            discretised_df[col] = discretised_df[col].fillna("Unknown")

    print("Data types after discretization:")
    print(discretised_df.dtypes.value_counts())

    print("\nUnique states per column:")
    print(discretised_df.nunique())

    print("\nDiscretised data preview:")
    print(discretised_df.head())

    if discretised_df.isna().any().any():
        print("\n⚠️ Warning: Some NaN values remain!")
        print(discretised_df.isna().sum()[discretised_df.isna().sum() > 0])
    
    return discretised_df

In [6]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

df_se = df_se.drop(["Unnamed: 0"], axis = 1)

df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,44430.0,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,43680.0,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,41470.0,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,43480.0,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,44510.0,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,44010.0,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,44130.0,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,44690.0,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,46160.0,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,46590.0,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [7]:
se = pd.DataFrame()
se = df_se.copy()
se['EUPC_2'] = se['EUPC'].shift(2)

In [8]:
se = se.dropna()

In [9]:
discretised_se = discretize_data(se)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
EUPC_2    3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP   QOA     QOR     YUA REST    EBSG    GGFC    LBGD  \
2  Low  Medium  Low  High    High    High  Low    High  Medium  Medium   
3  Low    High  Low  High    High    High  Low    High     Low  Medium   
4  Low    High  Low  High    High  Medium  Low    High     Low  Medium   
5  Low    High  Low  High    High    High  Low    High  Medium     Low   
6  Low     Low  Low   Low  Medium  Medium  Low  Medium    High     Low   

      FDI      PC     ARP  CO2C   PAM    EUPC  EUPC_2  
2  Medium  Medium     Low  High  High  Medium     Low  
3     Low  Medium     Low  High  High  Medium     Low  
4  Medium     Low     Low  High  High    Hig

In [10]:
sm = StructureModel()

In [11]:
sm.add_edges_from([
    ('EUPC_2', 'EBSG'),
    ('EUPC_2', 'QOR'),
])

In [12]:
sm.edges

OutEdgeView([('EUPC_2', 'EBSG'), ('EUPC_2', 'QOR')])

In [13]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [14]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_se)
bn_baseline.fit_cpds(discretised_se, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_se)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC_2  -> EBSG)
# ==========================================
print("\n--- Testing EUPC_2 -> EBSG ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC_2', 'EBSG'], 
    discretised_data=discretised_se,
    raw_data=se
)

# ==========================================
# TEST #2: (EUPC_2-> QOR)
# ==========================================
print("\n--- Testing EUPC -> QOR ---")

sm_test_2 = sm_baseline.copy()

bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_2,
    children_cols=['EUPC', 'QOR'], 
    discretised_data=discretised_se,
    raw_data=se
)

BASELINE BIC: 112.54

--- Testing EUPC_2 -> EBSG ---
Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'EBSG']
  - Removing direct edge: EUPC_2 -> EBSG
LV Model BIC: 130.06 (LogLikelihood: -45.24)

--- Testing EUPC -> QOR ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'QOR']
LV Model BIC: 186.14 (LogLikelihood: -60.08)


In [16]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

df_se = df_se.drop(["Unnamed: 0"], axis = 1)

df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,44430.0,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,43680.0,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,41470.0,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,43480.0,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,44510.0,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,44010.0,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,44130.0,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,44690.0,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,46160.0,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,46590.0,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [17]:
se = pd.DataFrame()
se = df_se.copy()
se['QOA_2'] = se['QOA'].shift(2)
se['GGFC_3'] = se['GGFC'].shift(3)

In [18]:
se = se.dropna()

In [19]:
discretised_se = discretize_data(se)

Data types after discretization:
object    18
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       2
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
QOA_2     3
GGFC_3    3
dtype: int64

Discretised data preview:
  GDPC  VABI  EMP   QOA   QOR     YUA REST    EBSG    GGFC    LBGD     FDI  \
3  Low  High  Low  High  High    High  Low    High     Low  Medium     Low   
4  Low  High  Low  High  High  Medium  Low    High     Low  Medium  Medium   
5  Low  High  Low  High  High    High  Low    High  Medium     Low  Medium   
6  Low   Low  Low   Low   Low  Medium  Low  Medium    High     Low     Low   
7  Low   Low  Low   Low   Low  Medium  Low  Medium    High     Low     Low   

       PC     ARP    CO2C     PAM    EUPC   QOA_2  GGFC_3  
3  Medium     Low    High    High  Medium    High     Low  
4     Low     Low    High    High    High    High     Lo

In [20]:
sm = StructureModel()

In [21]:
sm.add_edges_from([
    ('EUPC', 'FDI'),
    ('QOA_2', 'EUPC'),
    ('GGFC_3', 'EUPC')
])

In [22]:
sm.edges

OutEdgeView([('EUPC', 'FDI'), ('QOA_2', 'EUPC'), ('GGFC_3', 'EUPC')])

In [23]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [25]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_se)
bn_baseline.fit_cpds(discretised_se, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_se)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC  -> FDI)
# ==========================================
print("\n--- Testing EUPC -> FDI ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC', 'FDI'], 
    discretised_data=discretised_se,
    raw_data=se
)

# ==========================================
# TEST #2: (QOA_2 -> EUPC)
# ==========================================
print("\n--- Testing QOA_2 -> EUPC ---")

sm_test_2 = sm_baseline.copy()

bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_2,
    children_cols=['QOA_2', 'EUPC'], 
    discretised_data=discretised_se,
    raw_data=se
)

# ==========================================
# TEST #3: (GGFC_3 -> EUPC)
# ==========================================
print("\n--- Testing GGFC_3 -> EUPC ---")

sm_test_3 = sm_baseline.copy()

bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_3,
    children_cols=['GGFC_3', 'EUPC'], 
    discretised_data=discretised_se,
    raw_data=se
)



BASELINE BIC: 170.56

--- Testing EUPC -> FDI ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'FDI']
  - Removing direct edge: EUPC -> FDI
LV Model BIC: 228.41 (LogLikelihood: -56.49)

--- Testing QOA_2 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['QOA_2', 'EUPC']
  - Removing direct edge: QOA_2 -> EUPC
LV Model BIC: 173.88 (LogLikelihood: -54.88)

--- Testing GGFC_3 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['GGFC_3', 'EUPC']
  - Removing direct edge: GGFC_3 -> EUPC
LV Model BIC: 174.16 (LogLikelihood: -55.02)


In [26]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

df_se = df_se.drop(["Unnamed: 0"], axis = 1)

df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,44430.0,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,43680.0,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,41470.0,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,43480.0,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,44510.0,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,44010.0,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,44130.0,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,44690.0,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,46160.0,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,46590.0,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [27]:
se = pd.DataFrame()
se = df_se.copy()
se['EUPC_1'] = se['EUPC'].shift(1)

In [28]:
se = se.dropna()

In [29]:
discretised_se = discretize_data(se)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
EUPC_1    3
dtype: int64

Discretised data preview:
     GDPC    VABI     EMP   QOA     QOR     YUA REST  EBSG    GGFC    LBGD  \
1     Low    High  Medium  High    High     Low  Low  High     Low    High   
2     Low  Medium     Low  High    High    High  Low  High  Medium  Medium   
3     Low    High     Low  High    High    High  Low  High     Low  Medium   
4  Medium    High     Low  High    High  Medium  Low  High     Low  Medium   
5     Low    High     Low  High  Medium    High  Low  High  Medium     Low   

      FDI      PC  ARP  CO2C     PAM    EUPC  EUPC_1  
1    High  Medium  Low  High    High     Low     Low  
2  Medium  Medium  Low  High    High  Medium     Low  
3     Low  Medium  Low 

In [30]:
sm = StructureModel()

In [31]:
sm.add_edges_from([
    ('EUPC_1', 'PC')
])

In [32]:
sm.edges

OutEdgeView([('EUPC_1', 'PC')])

In [33]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [34]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_se)
bn_baseline.fit_cpds(discretised_se, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_se)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUP_1  -> PC)
# ==========================================
print("\n--- Testing EUPC_1 -> PC ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC_1', 'PC'], 
    discretised_data=discretised_se,
    raw_data=se
)



BASELINE BIC: 83.03

--- Testing EUPC_1 -> PC ---
Estimating LV based on continuous GMM distributions in: ['EUPC_1', 'PC']
  - Removing direct edge: EUPC_1 -> PC
LV Model BIC: 96.31 (LogLikelihood: -35.97)
